In [ ]:
import arcpy
import os
import csv

# =========================
# 0) Parameters
# =========================
arcpy.env.overwriteOutput = True
scratch_gdb = arcpy.env.scratchGDB

# Provincial boundary (including ADCODE99, NAME)
china_province = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\china_pro2.shp"

# ( name, gb)
county_fc = r"D:\\path\\to\\sinkhole-risk-china\\data\\Administrative_divisions_of_china\\Administrative_divisions_of_china\\county\\county.shp"

# Output CSV (province name in English_resolution)
province_en = "Guangdong"
resolution_km = 1
cell_size = 1000  # 3 km()

# ✅ county
out_csv = rf"D:\path\to\sinkhole-risk-china\data\output\county\Points_{province_en}_{resolution_km}km.csv"


# If you know Guangdong ADCODE99, you can fill it in directly (such as "440000"); if you don't fill it in, the script will automatically find "Guangdong" from the NAME
GUANGDONG_ADCODE99 = None

# =========================
# 1) :()
# =========================
def is_degree_sr(spref: arcpy.SpatialReference) -> bool:
    try:
        return (spref.type == "Geographic") or (spref.linearUnitName is None) or ("Degree" in (spref.linearUnitName or ""))
    except:
        return True

def find_field_case_insensitive(fc, candidates):
    """Find the field name in the feature class case-insensitively, return the actual field name or None"""
    fields = [f.name for f in arcpy.ListFields(fc)]
    low_map = {f.lower(): f for f in fields}
    for c in candidates:
        if c.lower() in low_map:
            return low_map[c.lower()]
    return None

# =========================
# 2) Check whether the necessary fields exist
# =========================
prov_ad = find_field_case_insensitive(china_province, ["ADCODE99"])
prov_name = find_field_case_insensitive(china_province, ["NAME"])
if prov_ad is None or prov_name is None:
    raise RuntimeError("Province boundary is missing ADCODE99 or NAME; check the china_pro2.shp fields.")

county_name = find_field_case_insensitive(county_fc, ["name", "NAME"])
county_gb = find_field_case_insensitive(county_fc, ["gb", "GB"])
if county_name is None or county_gb is None:
    raise RuntimeError("The county face is missing the field name or gb, please check the county face.shp field.")

# =========================
# 3) Find Guangdong ADCODE99 (if not filled in by hand)
# =========================
if GUANGDONG_ADCODE99 is None:
    gd_code = None
    with arcpy.da.SearchCursor(china_province, [prov_ad, prov_name]) as cur:
        for ad, nm in cur:
            if nm and ("Guangdong" in str(nm)):
                gd_code = ad
                break
    if gd_code is None:
        raise RuntimeError("No record containing \"Guangdong\" was found in the province boundary NAME field. Please set GUANGDONG_ADCODE99 manually.")
    GUANGDONG_ADCODE99 = gd_code

print(f"Guangdong ADCODE99 ={GUANGDONG_ADCODE99}")

# =========================
# 4)
# =========================
gd_layer = "gd_layer"
arcpy.management.MakeFeatureLayer(china_province, gd_layer)

ad_field_obj = [f for f in arcpy.ListFields(china_province) if f.name == prov_ad][0]
if ad_field_obj.type in ("String", "Guid"):
    where = f"{prov_ad} = '{GUANGDONG_ADCODE99}'"
else:
    where = f"{prov_ad} = {GUANGDONG_ADCODE99}"

arcpy.management.SelectLayerByAttribute(gd_layer, "NEW_SELECTION", where)

gd_poly = os.path.join(scratch_gdb, "Guangdong_polygon")
arcpy.management.CopyFeatures(gd_layer, gd_poly)

# =========================
# 5) (3km)
# =========================
sr_in = arcpy.Describe(gd_poly).spatialReference

try:
    sr_grid = arcpy.SpatialReference("Asia North Albers Equal Area Conic")
except:
    sr_grid = arcpy.SpatialReference(3857)  # Full details (unit: meters)

gd_for_grid = gd_poly
if is_degree_sr(sr_in):
    gd_for_grid = os.path.join(scratch_gdb, "Guangdong_projected_for_grid")
    arcpy.management.Project(gd_poly, gd_for_grid, sr_grid)
else:
    sr_grid = sr_in

# ,
county_for_grid = county_fc
county_sr = arcpy.Describe(county_fc).spatialReference
if county_sr.name != sr_grid.name:
    county_for_grid = os.path.join(scratch_gdb, "County_projected_for_grid")
    arcpy.management.Project(county_fc, county_for_grid, sr_grid)

arcpy.env.outputCoordinateSystem = sr_grid
gd_extent = arcpy.Describe(gd_for_grid).extent

# =========================
# 6) Create a 3km fishing net + points must fall within the plane (INSIDE) + crop to Guangdong
# =========================
print("Create a 3km fishing net and generate in-plane points (INSIDE)...")

fishnet = os.path.join(scratch_gdb, "fishnet_3km_poly")
origin_coord = f"{gd_extent.XMin} {gd_extent.YMin}"
y_axis_coord = f"{gd_extent.XMin} {gd_extent.YMax}"
corner_coord = f"{gd_extent.XMax} {gd_extent.YMax}"

arcpy.management.CreateFishnet(
    out_feature_class=fishnet,
    origin_coord=origin_coord,
    y_axis_coord=y_axis_coord,
    cell_width=cell_size,
    cell_height=cell_size,
    number_rows=0,
    number_columns=0,
    corner_coord=corner_coord,
    labels="NO_LABELS",
    template="#",
    geometry_type="POLYGON"
)

# ✅ The point must fall within the surface: INSIDE
fishnet_points = os.path.join(scratch_gdb, "fishnet_3km_points_inside_cell")
arcpy.management.FeatureToPoint(fishnet, fishnet_points, "INSIDE")

# ✅ Crop to Guangdong: Make sure the point is within Guangdong Province
gd_points_proj = os.path.join(scratch_gdb, "Guangdong_3km_points_projected")
arcpy.analysis.Clip(fishnet_points, gd_for_grid, gd_points_proj)

# =========================
# 7) Spatial Join: First add the province attribute (ADCODE99), then add the county attribute (name/gb)
# =========================
print("Overlaying provincial attributes and county attributes...")

# 7.1 Provincial attributes
gd_points_with_prov = os.path.join(scratch_gdb, "Guangdong_3km_points_with_prov")
arcpy.analysis.SpatialJoin(
    target_features=gd_points_proj,
    join_features=gd_for_grid,
    out_feature_class=gd_points_with_prov,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    match_option="INTERSECT"
)

# 7.2 County attributes
gd_points_with_county = os.path.join(scratch_gdb, "Guangdong_3km_points_with_county")
arcpy.analysis.SpatialJoin(
    target_features=gd_points_with_prov,
    join_features=county_for_grid,
    out_feature_class=gd_points_with_county,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    match_option="INTERSECT"
)

# =========================
# 8) Project to WGS84 and add fields Province / Longitude / Latitude
# =========================
print("Project to WGS84 and calculate latitude and longitude...")
wgs84 = arcpy.SpatialReference(4326)
gd_points_wgs84 = os.path.join(scratch_gdb, "Guangdong_3km_points_WGS84")
arcpy.management.Project(gd_points_with_county, gd_points_wgs84, wgs84)

# Add fields Province / Longitude / Latitude
fnames = [f.name for f in arcpy.ListFields(gd_points_wgs84)]
if "Province" not in fnames:
    arcpy.management.AddField(gd_points_wgs84, "Province", "TEXT", field_length=100)
if "Longitude" not in fnames:
    arcpy.management.AddField(gd_points_wgs84, "Longitude", "DOUBLE")
if "Latitude" not in fnames:
    arcpy.management.AddField(gd_points_wgs84, "Latitude", "DOUBLE")

with arcpy.da.UpdateCursor(gd_points_wgs84, ["SHAPE@X", "SHAPE@Y", "Province", "Longitude", "Latitude"]) as cur:
    for x, y, prov, lon, lat in cur:
        cur.updateRow((x, y, province_en, x, y))

# =========================
# 9) Identify the field name after join (may contain _1)
# =========================
fnames = [f.name for f in arcpy.ListFields(gd_points_wgs84)]

# Provincial ADC field
ad_out = "ADCODE99" if "ADCODE99" in fnames else ("ADCODE99_1" if "ADCODE99_1" in fnames else None)
if ad_out is None:
    # Sometimes the original field name (prov_ad) will be retained and the details will be revealed again.
    ad_out = prov_ad if prov_ad in fnames else None
if ad_out is None:
    raise RuntimeError("ADCODE99 field (or its _1 version) not found in output point, please check Spatial Join output.")

# County field
cname_out = county_name if county_name in fnames else (f"{county_name}_1" if f"{county_name}_1" in fnames else None)
cgb_out = county_gb if county_gb in fnames else (f"{county_gb}_1" if f"{county_gb}_1" in fnames else None)
if cname_out is None or cgb_out is None:
    raise RuntimeError("The county field name/gb (or its _1 version) was not found in the output point, please check the Spatial Join output.")

# =========================
# 10) CSV:No, ADCODE99, Province, County, gb, Longitude, Latitude
# =========================
print("CSV...")
out_dir = os.path.dirname(out_csv)
os.makedirs(out_dir, exist_ok=True)

rows = 0
with open(out_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["No", "ADCODE99", "Province", "County", "gb", "Longitude", "Latitude"])

    with arcpy.da.SearchCursor(
        gd_points_wgs84,
        [ad_out, "Province", cname_out, cgb_out, "Longitude", "Latitude"]
    ) as cur:
        i = 1
        for ad, prov, cname, cgb, lon, lat in cur:
            writer.writerow([i, ad, prov, cname, cgb, lon, lat])
            i += 1
            rows += 1

print(f"Done! Guangdong{resolution_km}km Number of grid points:{rows}")
print(f"CSV saved:{out_csv}")


The following is the code to generate 34 areas at one time

In [ ]:
import arcpy
import os
import csv
import re

# =========================
# 0) Parameters
# =========================
arcpy.env.overwriteOutput = True
scratch_gdb = arcpy.env.scratchGDB

china_province = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\china_pro2.shp"
county_fc = r"D:\\path\\to\\sinkhole-risk-china\\data\\Administrative_divisions_of_china\\Administrative_divisions_of_china\\county\\county.shp"

resolution_km = 1
cell_size = 1000  # 3km (meters)

out_dir = r"D:\path\to\sinkhole-risk-china\data\output\county"

# =========================
# 1) -> (/Province)
# =========================
PROVINCE_EN_MAP = {
    "Beijing": "Beijing",
    "Tianjin": "Tianjin",
    "Hebei": "Hebei",
    "Shanxi": "Shanxi",
    "Inner Mongolia": "InnerMongolia",
    "Liaoning": "Liaoning",
    "Jilin": "Jilin",
    "Heilongjiang": "Heilongjiang",
    "Shanghai": "Shanghai",
    "Jiangsu": "Jiangsu",
    "Zhejiang": "Zhejiang",
    "Anhui": "Anhui",
    "Fujian": "Fujian",
    "Jiangxi": "Jiangxi",
    "Shandong": "Shandong",
    "Henan": "Henan",
    "Hubei": "Hubei",
    "Hunan": "Hunan",
    "Guangdong": "Guangdong",
    "Guangxi": "Guangxi",
    "Hainan": "Hainan",
    "Chongqing": "Chongqing",
    "Sichuan": "Sichuan",
    "Guizhou": "Guizhou",
    "Yunnan": "Yunnan",
    "Tibet": "Tibet",
    "Shaanxi": "Shaanxi",
    "Gansu": "Gansu",
    "Qinghai": "Qinghai",
    "Ningxia": "Ningxia",
    "Xinjiang": "Xinjiang",
    "HongKong": "HongKong",
    "Macau": "Macau",
    "Taiwan": "Taiwan",
}

# =========================
# 2) Tool function
# =========================
def is_degree_sr(spref: arcpy.SpatialReference) -> bool:
    try:
        return (spref.type == "Geographic") or (spref.linearUnitName is None) or ("Degree" in (spref.linearUnitName or ""))
    except:
        return True

def find_field_case_insensitive(fc, candidates):
    fields = [f.name for f in arcpy.ListFields(fc)]
    low_map = {f.lower(): f for f in fields}
    for c in candidates:
        if c.lower() in low_map:
            return low_map[c.lower()]
    return None

def safe_filename(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r'[\\/:*?"<>|]+', "_", name)
    name = re.sub(r"\s+", "_", name)
    return name

def normalize_province_cn(name_cn: str) -> str:
    """NAME , dict key(://///). :////// ."""
    if name_cn is None:
        return ""
    s = str(name_cn).strip()
    # Common suffix/word removal
    for k in ["Province", "City", "Autonomous Region", "Special Administrative Region", "Zhuang", "Hui", "Uyghur", "Uygur"]:
        s = s.replace(k, "")
    s = s.replace("Xinjiang Uygur", "Xinjiang")  # Normalize alternate Xinjiang naming.
    s = s.replace("Inner Mongolia", "Inner Mongolia")
    s = s.replace("Guangxi", "Guangxi")
    s = s.replace("Ningxia", "Ningxia")
    s = s.replace("Tibet", "Tibet")
    return s

def province_cn_to_en(name_cn: str):
    """; name_cn None()."""
    if name_cn is None:
        return None
    s = str(name_cn).strip()
    if s == "":
        return None

    key = normalize_province_cn(s)
    if key in PROVINCE_EN_MAP:
        return PROVINCE_EN_MAP[key]

    raise RuntimeError(f"Failed to map province name to English: NAME='{name_cn}' ='{key}'. Please supplement PROVINCE_EN_MAP or adjust the normalization rules.")

# =========================
# 3) Field check
# =========================
prov_ad = find_field_case_insensitive(china_province, ["ADCODE99"])
prov_name = find_field_case_insensitive(china_province, ["NAME"])
if prov_ad is None or prov_name is None:
    raise RuntimeError("Missing field ADCODE99 or NAME for province boundary, please check china_pro2.shp.")

county_name = find_field_case_insensitive(county_fc, ["name", "NAME"])
county_gb = find_field_case_insensitive(county_fc, ["gb", "GB"])
if county_name is None or county_gb is None:
    raise RuntimeError("The county face is missing the field name or gb, please check the county face.shp.")

# =========================
# 4) Unified projected coordinate system (used to generate 3km grid)
# =========================
try:
    sr_grid_default = arcpy.SpatialReference("Asia North Albers Equal Area Conic")
except:
    sr_grid_default = arcpy.SpatialReference(3857)

sr_prov_in = arcpy.Describe(china_province).spatialReference
china_prov_for_grid = china_province
sr_grid = sr_prov_in

if is_degree_sr(sr_prov_in):
    sr_grid = sr_grid_default
    china_prov_for_grid = os.path.join(scratch_gdb, "china_province_projected_for_grid")
    arcpy.management.Project(china_province, china_prov_for_grid, sr_grid)

county_for_grid = county_fc
sr_county_in = arcpy.Describe(county_fc).spatialReference
if sr_county_in.name != sr_grid.name:
    county_for_grid = os.path.join(scratch_gdb, "county_projected_for_grid")
    arcpy.management.Project(county_fc, county_for_grid, sr_grid)

arcpy.env.outputCoordinateSystem = sr_grid
os.makedirs(out_dir, exist_ok=True)

# =========================
# 5) Summarize provincial records (remove duplication according to ADCODE99)
# =========================
province_records = {}
with arcpy.da.SearchCursor(china_prov_for_grid, [prov_ad, prov_name]) as cur:
    for ad, name_cn in cur:
        if ad not in province_records:
            province_records[ad] = name_cn

print(f"Number of provincial units will be processed:{len(province_records)}")

# =========================
# 6) Traverse each province: generate 3km INSIDE points + overlay counties + output CSV
# =========================
wgs84 = arcpy.SpatialReference(4326)

for idx, (adcode, name_cn) in enumerate(province_records.items(), start=1):
    prov_en = province_cn_to_en(name_cn)   # ✅
    prov_en_safe = safe_filename(prov_en)

    out_csv = os.path.join(out_dir, f"Points_{prov_en_safe}_{resolution_km}km.csv")
    print(f"\n[{idx}/{len(province_records)}] Processing:{name_cn} -> {prov_en} (ADCODE99={adcode})")
    print(f":{out_csv}")

    # 6.1 Select the province
    lyr = "prov_lyr"
    arcpy.management.MakeFeatureLayer(china_prov_for_grid, lyr)

    ad_field_obj = [f for f in arcpy.ListFields(china_prov_for_grid) if f.name == prov_ad][0]
    if ad_field_obj.type in ("String", "Guid"):
        where = f"{prov_ad} = '{adcode}'"
    else:
        where = f"{prov_ad} = {adcode}"

    arcpy.management.SelectLayerByAttribute(lyr, "NEW_SELECTION", where)

    prov_poly = os.path.join(scratch_gdb, f"prov_poly_{idx}")
    arcpy.management.CopyFeatures(lyr, prov_poly)

    # 6.2 Create a 3km fishing net + points must fall within the plane (INSIDE) + Clip to the provincial plane
    ext = arcpy.Describe(prov_poly).extent

    fishnet = os.path.join(scratch_gdb, f"fishnet_{idx}")
    origin_coord = f"{ext.XMin} {ext.YMin}"
    y_axis_coord = f"{ext.XMin} {ext.YMax}"
    corner_coord = f"{ext.XMax} {ext.YMax}"

    arcpy.management.CreateFishnet(
        out_feature_class=fishnet,
        origin_coord=origin_coord,
        y_axis_coord=y_axis_coord,
        cell_width=cell_size,
        cell_height=cell_size,
        number_rows=0,
        number_columns=0,
        corner_coord=corner_coord,
        labels="NO_LABELS",
        template="#",
        geometry_type="POLYGON"
    )

    fishnet_points = os.path.join(scratch_gdb, f"fishnet_pts_{idx}")
    arcpy.management.FeatureToPoint(fishnet, fishnet_points, "INSIDE")  # ✅ Must fall within the surface

    prov_points_proj = os.path.join(scratch_gdb, f"prov_pts_clip_{idx}")
    arcpy.analysis.Clip(fishnet_points, prov_poly, prov_points_proj)

    # 6.3 Spatial Join: Province attribute + County attribute
    pts_with_prov = os.path.join(scratch_gdb, f"pts_with_prov_{idx}")
    arcpy.analysis.SpatialJoin(
        target_features=prov_points_proj,
        join_features=prov_poly,
        out_feature_class=pts_with_prov,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_COMMON",
        match_option="INTERSECT"
    )

    pts_with_county = os.path.join(scratch_gdb, f"pts_with_county_{idx}")
    arcpy.analysis.SpatialJoin(
        target_features=pts_with_prov,
        join_features=county_for_grid,
        out_feature_class=pts_with_county,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_COMMON",
        match_option="INTERSECT"
    )

    # 6.4 Project to WGS84 + calculate latitude and longitude + write Province (English)
    pts_wgs84 = os.path.join(scratch_gdb, f"pts_wgs84_{idx}")
    arcpy.management.Project(pts_with_county, pts_wgs84, wgs84)

    fnames = [f.name for f in arcpy.ListFields(pts_wgs84)]
    if "Province" not in fnames:
        arcpy.management.AddField(pts_wgs84, "Province", "TEXT", field_length=100)
    if "Longitude" not in fnames:
        arcpy.management.AddField(pts_wgs84, "Longitude", "DOUBLE")
    if "Latitude" not in fnames:
        arcpy.management.AddField(pts_wgs84, "Latitude", "DOUBLE")

    with arcpy.da.UpdateCursor(pts_wgs84, ["SHAPE@X", "SHAPE@Y", "Province", "Longitude", "Latitude"]) as cur2:
        for x, y, provv, lon, lat in cur2:
            cur2.updateRow((x, y, prov_en, x, y))

    # 6.5 Identify the field name after join (possibly _1)
    fnames = [f.name for f in arcpy.ListFields(pts_wgs84)]

    ad_out = prov_ad if prov_ad in fnames else (f"{prov_ad}_1" if f"{prov_ad}_1" in fnames else None)
    if ad_out is None:
        ad_out = "ADCODE99" if "ADCODE99" in fnames else ("ADCODE99_1" if "ADCODE99_1" in fnames else None)
    if ad_out is None:
        raise RuntimeError(f"[{prov_en}] ADCODE99 field (or _1) not found in output point.")

    cname_out = county_name if county_name in fnames else (f"{county_name}_1" if f"{county_name}_1" in fnames else None)
    cgb_out = county_gb if county_gb in fnames else (f"{county_gb}_1" if f"{county_gb}_1" in fnames else None)
    if cname_out is None or cgb_out is None:
        raise RuntimeError(f"[{prov_en}] name/gb( _1).")

    # 6.6 Output CSV
    with open(out_csv, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["No", "ADCODE99", "Province", "County", "gb", "Longitude", "Latitude"])

        i = 1
        with arcpy.da.SearchCursor(pts_wgs84, [ad_out, "Province", cname_out, cgb_out, "Longitude", "Latitude"]) as cur3:
            for ad_v, prov_v, county_v, gb_v, lon_v, lat_v in cur3:
                writer.writerow([i, ad_v, prov_v, county_v, gb_v, lon_v, lat_v])
                i += 1

    npts = int(arcpy.management.GetCount(pts_wgs84)[0])
    print(f"Completed:{prov_en}-> Score{npts}")

print("\\n All provinces are processed!")
